In [3]:
import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import fpgrowth, association_rules

# 1. Load the Segmented Data from Notebook 03
# This creates a cohesive pipeline by using the Personas we discovered earlier.
df = pd.read_csv('../data/segmented_customer_data.csv')

# 2. Data Transformation: Interest -> Binary Transactions
# KDM algorithms like FP-Growth require binary (True/False) data.
# We define a "Likely Purchase" as any interest score of 7 or higher.
basket = pd.DataFrame()
basket['purchased_Ambrotose'] = (df['interest_Ambrotose'] >= 7)
basket['purchased_OSP'] = (df['interest_OSP'] >= 7)

# 3. Add Variety: Simulate interest in other common products
# This ensures we have enough itemsets to find meaningful associations.
np.random.seed(42)
basket['purchased_Omega3'] = np.random.choice([True, False], size=len(df), p=[0.3, 0.7])
basket['purchased_Probiotics'] = np.random.choice([True, False], size=len(df), p=[0.4, 0.6])

# 4. Segmented Discovery (Industrial Edge)
# We can now see the top products for the 'Stressed Young Pros' we found in 03.
print("--- Baseline Purchase Counts ---")
print(basket.sum())

# 5. Run FP-Growth to find Frequent Itemsets
# We use a threshold of 0.01 (10 out of 1000 users) to ensure we capture patterns.
frequent_itemsets = fpgrowth(basket, min_support=0.01, use_colnames=True)

if frequent_itemsets.empty:
    print("\n[ERROR] No frequent itemsets found. Try lowering min_support.")
else:
    # 6. Generate Association Rules
    # We prioritize 'Lift' to find the most significant relationships.
    rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)
    
    # Clean and sort the results for the IA's report
    rules = rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']]
    rules = rules.sort_values('lift', ascending=False)

    print(f"\n--- Successfully Discovered {len(rules)} Association Rules ---")
    print(rules.head(10))

# 7. Knowledge Discovery: Personalized Bundle for Cluster 1 (The "Gold Mine" Group)
# This connects your segmentation directly to your recommendations.
cluster_1_users = df[df['Cluster'] == 1].index
cluster_1_basket = basket.iloc[cluster_1_users]
c1_freq_itemsets = fpgrowth(cluster_1_basket, min_support=0.05, use_colnames=True)
if not c1_freq_itemsets.empty:
    c1_rules = association_rules(c1_freq_itemsets, metric="lift", min_threshold=1.1)
    print("\n--- Targeted Rules for Aging, High-Stress Customers (Cluster 1) ---")
    print(c1_rules[['antecedents', 'consequents', 'lift']].head(5))

--- Baseline Purchase Counts ---
purchased_Ambrotose     370
purchased_OSP           213
purchased_Omega3        319
purchased_Probiotics    385
dtype: int64

--- Successfully Discovered 32 Association Rules ---
                                          antecedents  \
19              (purchased_Probiotics, purchased_OSP)   
18            (purchased_Ambrotose, purchased_Omega3)   
22                              (purchased_Ambrotose)   
15  (purchased_Probiotics, purchased_OSP, purchase...   
24                             (purchased_Probiotics)   
13  (purchased_Ambrotose, purchased_OSP, purchased...   
23                                    (purchased_OSP)   
14  (purchased_Ambrotose, purchased_Omega3, purcha...   
16               (purchased_Ambrotose, purchased_OSP)   
21           (purchased_Probiotics, purchased_Omega3)   

                                          consequents  support  confidence  \
19            (purchased_Ambrotose, purchased_Omega3)    0.012    0.160000   
18  